המחברת הבאה נוצרה אוטומטית על ידי GitHub Copilot Chat ומיועדת רק להגדרה ראשונית


# מבוא להנדסת פרומפטים
הנדסת פרומפטים היא תהליך של תכנון ואופטימיזציה של פרומפטים למשימות עיבוד שפה טבעית. היא כוללת בחירת הפרומפטים הנכונים, כוונון הפרמטרים שלהם והערכת הביצועים שלהם. הנדסת פרומפטים היא חיונית להשגת דיוק ויעילות גבוהים במודלים לעיבוד שפה טבעית. בפרק זה נחקור את היסודות של הנדסת פרומפטים באמצעות מודלי OpenAI לצורך חקירה.


### תרגיל 1: הטוקניזציה
חקור את הטוקניזציה באמצעות tiktoken, טוקנייזר מהיר בקוד פתוח מ-OpenAI
ראה [OpenAI Cookbook](https://github.com/openai/openai-cookbook/blob/main/examples/How_to_count_tokens_with_tiktoken.ipynb?WT.mc_id=academic-105485-koreyst) לעוד דוגמאות.


In [ ]:
# EXERCISE:
# 1. Run the exercise as is first
# 2. Change the text to any prompt input you want to use & re-run to see tokens

import tiktoken

# Define the prompt you want tokenized
text = f"""
Jupiter is the fifth planet from the Sun and the \
largest in the Solar System. It is a gas giant with \
a mass one-thousandth that of the Sun, but two-and-a-half \
times that of all the other planets in the Solar System combined. \
Jupiter is one of the brightest objects visible to the naked eye \
in the night sky, and has been known to ancient civilizations since \
before recorded history. It is named after the Roman god Jupiter.[19] \
When viewed from Earth, Jupiter can be bright enough for its reflected \
light to cast visible shadows,[20] and is on average the third-brightest \
natural object in the night sky after the Moon and Venus.
"""

# Set the model you want encoding for
encoding = tiktoken.encoding_for_model("gpt-4o")

# Encode the text - gives you the tokens in integer form
tokens = encoding.encode(text)
print(tokens);

# Decode the integers to see what the text versions look like
[encoding.decode_single_token_bytes(token) for token in tokens]


### תרגיל 2: אימות הגדרת מפתח ה-API של OpenAI

הפעל את הקוד למטה כדי לוודא שנקודת הקצה של OpenAI הוגדרה נכון. הקוד פשוט מנסה הפעלה פשוטה ובסיסית ומאמת את ההשלמה. הקלט `oh say can you see` אמור להסתיים בקירוב עם `by the dawn's early light..`


In [ ]:
# Uses the OpenAI client against the Azure OpenAI (Microsoft Foundry) v1 endpoint
# with the Responses API. See https://aka.ms/openai/start

import os
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

client = OpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],
  base_url=f"{os.environ['AZURE_OPENAI_ENDPOINT'].rstrip('/')}/openai/v1/",
  )

deployment=os.environ['AZURE_OPENAI_DEPLOYMENT']

def get_completion(prompt):
    response = client.responses.create(
        model=deployment,
        input=prompt,
        temperature=0, # this is the degree of randomness of the model's output
        max_output_tokens=1024,
        store=False,
    )
    return response.output_text

## ---------- Call the helper method

### 1. Set primary content or prompt text
text = f"""
oh say can you see
"""

### 2. Use that in the prompt template below
prompt = f"""
```{text}```
"""

## 3. Run the prompt
response = get_completion(prompt)
print(response)


### תרגיל 3: פבריקציות
חקר מה קורה כשאתה מבקש מה-LLM להחזיר השלמות עבור בקשה בנושא שייתכן שלא קיים, או בנושאים שאולי הוא לא מכיר כי הם היו מחוץ למאגר הנתונים עליו אומן מראש (חדשים יותר). ראה כיצד התגובה משתנה אם תנסה בקשה שונה, או מודל שונה.


In [ ]:

## Set the text for simple prompt or primary content
## Prompt shows a template format with text in it - add cues, commands etc if needed
## Run the completion 
text = f"""
generate a lesson plan on the Martian War of 2076.
"""

prompt = f"""
```{text}```
"""

response = get_completion(prompt)
print(response)

### תרגיל 4: מבוסס על הוראות
השתמשו במשתנה "text" כדי להגדיר את התוכן הראשי
ובמשתנה "prompt" כדי לספק הוראה הקשורה לתוכן הראשי.

כאן אנחנו מבקשים מהמודל לסכם את הטקסט עבור תלמיד בכיתה ב'


In [ ]:
# Test Example
# https://platform.openai.com/playground/p/default-summarize

## Example text
text = f"""
Jupiter is the fifth planet from the Sun and the \
largest in the Solar System. It is a gas giant with \
a mass one-thousandth that of the Sun, but two-and-a-half \
times that of all the other planets in the Solar System combined. \
Jupiter is one of the brightest objects visible to the naked eye \
in the night sky, and has been known to ancient civilizations since \
before recorded history. It is named after the Roman god Jupiter.[19] \
When viewed from Earth, Jupiter can be bright enough for its reflected \
light to cast visible shadows,[20] and is on average the third-brightest \
natural object in the night sky after the Moon and Venus.
"""

## Set the prompt
prompt = f"""
Summarize content you are provided with for a second-grade student.
```{text}```
"""

## Run the prompt
response = get_completion(prompt)
print(response)

### תרגיל 5: בקשה מורכבת  
נסה בקשה שכוללת הודעות של המערכת, המשתמש והעוזר  
המערכת קובעת את הקשר העוזר  
הודעות המשתמש והעוזר מספקות הקשר שיחה מרובה סבבים  

שים לב כיצד אישיות העוזר מוגדרת ל"סרקסטי" בהקשר המערכת.  
נסה להשתמש בהקשר אישיות שונה. או נסה סדר שונה של הודעות קלט/פלט  


In [ ]:
response = client.responses.create(
    model=deployment,
    input=[
        {"role": "system", "content": "You are a sarcastic assistant."},
        {"role": "user", "content": "Who won the world series in 2020?"},
        {"role": "assistant", "content": "Who do you think won? The Los Angeles Dodgers of course."},
        {"role": "user", "content": "Where was it played?"}
    ],
    store=False,
)
print(response.output_text)


### תרגיל: חקור את האינטואיציה שלך
הדוגמאות שלמעלה נותנות לך דפוסים שניתן להשתמש בהם ליצירת קריאות חדשות (פשוטות, מורכבות, הוראות וכו') - נסה ליצור תרגילים נוספים כדי לחקור כמה מהרעיונות האחרים שדיברנו עליהם כמו דוגמאות, רמזים ויותר.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**כתב ויתור**:
מסמך זה תורגם באמצעות שירות תרגום אוטומטי [Co-op Translator](https://github.com/Azure/co-op-translator). למרות שאנו שואפים לדיוק, יש לקחת בחשבון שתרגומים אוטומטיים עלולים להכיל שגיאות או אי-דיוקים. יש להחשיב את המסמך המקורי בשפתו הטבעית כמקור הסמכות. למידע קריטי מומלץ להשתמש בתרגום מקצועי על ידי מתרגם אדם. אנו לא אחראים לכל אי-הבנה או פירוש שגוי הנובע מהשימוש בתרגום זה.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
